# Segmentación y Grad-CAM con MONAI

Esta práctica usa Task09_Spleen, un conjunto de CT 3D con máscaras reales del bazo. Entrenaremos una U-Net 2D sobre cortes axiales seleccionados por volumen y compararemos máscara de referencia, predicción y Grad-CAM del canal de foreground. Con CUDA se usa el recorrido completo; en CPU el notebook activa `QUICK_RUN` para limitar volúmenes, cortes y épocas.

**Objetivos**
- Preparar una partición por volumen antes de extraer cortes, evitando fuga entre cortes del mismo estudio.
- Entrenar y evaluar una segmentación binaria con Dice.
- Aplicar Grad-CAM a los logits espaciales del canal bazo y discutir qué significa.

> Grad-CAM explica activaciones asociadas al score de foreground agregado; no es una máscara, no reemplaza Dice y no demuestra causalidad. El modelo es 2D slice-wise, no una segmentación volumétrica clínica.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

missing_packages = [
    package for package, module in (("monai", "monai"), ("nibabel", "nibabel"))
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from monai.apps import download_and_extract
from monai.data import DataLoader, Dataset
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.networks.nets import UNet
from monai.transforms import (
    Compose, EnsureChannelFirstd, EnsureTyped, LoadImaged,
    Orientationd, Resized, ScaleIntensityRanged,
)
from monai.utils import set_determinism
from monai.visualize import GradCAM

SEED = 31
set_determinism(seed=SEED)
if torch.cuda.is_available():
    device = torch.device("cuda")
    QUICK_RUN = False
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    QUICK_RUN = True
else:
    device = torch.device("cpu")
    QUICK_RUN = True
data_root = Path(os.environ.get("MONAI_DATA_DIRECTORY", Path.home() / ".cache" / "monai"))
data_root.mkdir(parents=True, exist_ok=True)
print("MONAI:", __import__("monai").__version__, "| dispositivo:", device)
print("Modo rápido:", QUICK_RUN)
print("Directorio persistente:", data_root)

## 1. Descargar Task09_Spleen

El archivo contiene imágenes NIfTI y máscaras de referencia. La primera descarga incluye el conjunto completo, por lo que puede tardar y ocupar espacio; `MONAI_DATA_DIRECTORY` permite reutilizarlo entre sesiones. La verificación MD5 detecta descargas incompletas o alteradas.

In [ ]:
resource = "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task09_Spleen.tar"
md5 = "410d4a301da4e5b2f6f86ec3ddba524e"
archive_path = data_root / "Task09_Spleen.tar"
data_dir = data_root / "Task09_Spleen"
if not data_dir.exists():
    download_and_extract(resource, str(archive_path), str(data_root), md5)

image_paths = sorted((data_dir / "imagesTr").glob("*.nii.gz"))
label_paths = sorted((data_dir / "labelsTr").glob("*.nii.gz"))
assert len(image_paths) == len(label_paths) and len(image_paths) > 0
cases = [{"image": str(image), "label": str(label)} for image, label in zip(image_paths, label_paths)]
print(f"Casos con máscara: {len(cases)}")
print("Los archivos imagesTs no tienen etiquetas públicas; la evaluación se hace en volúmenes etiquetados reservados.")

## 2. Separar por volumen y seleccionar cortes

Primero separamos los estudios completos: 32 volúmenes para entrenamiento y 9 para validación, siguiendo el patrón del notebook de segmentación de Unidad 2. Solo después elegimos cortes axiales. Así, dos cortes del mismo volumen no aparecen en ambos grupos. En modo rápido se usan pocos volúmenes, pero se conserva la misma regla.

In [ ]:
rng = np.random.default_rng(SEED)
case_order = rng.permutation(len(cases))
val_count = 9
val_indices = set(case_order[-val_count:].tolist())
train_cases = [case for index, case in enumerate(cases) if index not in val_indices]
val_cases = [case for index, case in enumerate(cases) if index in val_indices]
if QUICK_RUN:
    train_cases = train_cases[:4]
    val_cases = val_cases[:2]

print(f"Volúmenes train: {len(train_cases)} | validación: {len(val_cases)}")
print("Separación realizada por estudio antes de extraer slices.")

## 3. Transformaciones MONAI y dataset 2D

Cargamos CT y etiqueta juntos, orientamos ambos al mismo sistema y escalamos las intensidades de CT. Para el entrenamiento seleccionamos cortes con y sin bazo y redimensionamos imagen con interpolación bilineal y etiqueta con vecino más cercano. La máscara siempre conserva etiquetas discretas.

In [ ]:
volume_transform = Compose([
    LoadImaged(keys=("image", "label")),
    EnsureChannelFirstd(keys=("image", "label")),
    Orientationd(keys=("image", "label"), axcodes="RAS"),
    ScaleIntensityRanged(
        keys="image", a_min=-57, a_max=164, b_min=0.0, b_max=1.0, clip=True
    ),
])
slice_transform = Compose([
    Resized(keys=("image", "label"), spatial_size=(128, 128), mode=("bilinear", "nearest")),
    EnsureTyped(keys="image", dtype=torch.float32),
    EnsureTyped(keys="label", dtype=torch.long),
])

def volume_to_slices(case, max_positive, max_negative, seed):
    volume = volume_transform(case)
    image_volume = volume["image"].as_tensor()[0].cpu().numpy()
    label_volume = volume["label"].as_tensor()[0].cpu().numpy()
    positive_slices = np.flatnonzero((label_volume > 0).sum(axis=(0, 1)) > 0)
    negative_slices = np.flatnonzero((label_volume > 0).sum(axis=(0, 1)) == 0)
    local_rng = np.random.default_rng(seed)
    selected = []
    for candidates, limit in ((positive_slices, max_positive), (negative_slices, max_negative)):
        if len(candidates):
            selected.extend(local_rng.choice(candidates, min(limit, len(candidates)), replace=False).tolist())
    records = []
    for slice_index in sorted(selected):
        records.append({
            "image": image_volume[:, :, slice_index][None].astype(np.float32),
            "label": (label_volume[:, :, slice_index] > 0).astype(np.int64)[None],
        })
    return records

positive_per_volume = 6 if QUICK_RUN else 12
negative_per_volume = 2 if QUICK_RUN else 4
train_slices = [
    record
    for index, case in enumerate(train_cases)
    for record in volume_to_slices(case, positive_per_volume, negative_per_volume, SEED + index)
]
val_slices = [
    record
    for index, case in enumerate(val_cases)
    for record in volume_to_slices(case, positive_per_volume, negative_per_volume, SEED + 100 + index)
]
train_ds = Dataset(data=train_slices, transform=slice_transform)
train_loader = DataLoader(
    train_ds,
    batch_size=4 if QUICK_RUN else 16,
    shuffle=True,
    num_workers=0,
)
print(f"Slices train: {len(train_ds)} | slices de validación para inspección: {len(val_slices)}")

sample = slice_transform(train_slices[0])
fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(sample["image"][0], cmap="gray"); axes[0].set_title("CT")
axes[1].imshow(sample["label"][0]); axes[1].set_title("Máscara bazo")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## 4. U-Net 2D y métrica Dice

La U-Net recibe un corte axial y predice dos canales: fondo y bazo. La función Dice+CE combina solapamiento y clasificación voxel a voxel. En modo rápido se entrenan dos épocas con pocos volúmenes; la opción completa usa todos los volúmenes de entrenamiento y más cortes por caso.

In [ ]:
model = UNet(
    spatial_dims=2, in_channels=1, out_channels=2,
    channels=(8, 16, 32, 64), strides=(2, 2, 2), num_res_units=1,
).to(device)
loss_function = DiceCELoss(to_onehot_y=True, softmax=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
NUM_EPOCHS = 2 if QUICK_RUN else 8

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = loss_function(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Época {epoch + 1}/{NUM_EPOCHS}: train loss={running_loss / len(train_loader):.4f}")

## 5. Evaluación por volumen retenido

La validación se hace sobre todos los cortes de cada volumen retenido, no sobre los cortes de entrenamiento. Redimensionamos CT con bilineal y máscara con vecino más cercano; calculamos Dice del foreground por volumen. En este ejemplo el modelo trabaja slice-wise: no aprende contexto entre cortes ni produce una salida 3D nativa.

In [ ]:
dice_metric = DiceMetric(include_background=False, reduction="mean")

@torch.no_grad()
def predict_volume(case, batch_size=8):
    volume = volume_transform(case)
    image_volume = volume["image"].as_tensor()[0].cpu().numpy()
    label_volume = (volume["label"].as_tensor()[0].cpu().numpy() > 0).astype(np.uint8)
    image_slices = torch.from_numpy(np.moveaxis(image_volume, -1, 0)[:, None]).float()
    label_slices = torch.from_numpy(np.moveaxis(label_volume, -1, 0)[:, None]).float()
    image_slices = F.interpolate(image_slices, size=(128, 128), mode="bilinear", align_corners=False)
    label_slices = F.interpolate(label_slices, size=(128, 128), mode="nearest").long()
    model.eval()
    predictions = []
    for start in range(0, len(image_slices), batch_size):
        logits = model(image_slices[start:start + batch_size].to(device))
        predictions.append(logits.argmax(dim=1).cpu())
    prediction = torch.cat(predictions)

    prediction_volume = prediction.permute(1, 2, 0).unsqueeze(0)
    target_volume = label_slices[:, 0].permute(1, 2, 0).unsqueeze(0)
    prediction_one_hot = F.one_hot(prediction_volume, num_classes=2).movedim(-1, 1).float()
    target_one_hot = F.one_hot(target_volume, num_classes=2).movedim(-1, 1).float()
    dice_metric(y_pred=prediction_one_hot, y=target_one_hot)
    dice = float(dice_metric.aggregate().item())
    dice_metric.reset()
    return image_slices, label_slices[:, 0], prediction, dice

validation_cases = val_cases[:1] if QUICK_RUN else val_cases
volume_dice = []
for case in validation_cases:
    _, _, _, score = predict_volume(case)
    volume_dice.append(score)
    print(f"{Path(case['image']).name}: Dice foreground por volumen={score:.4f}")
print(f"Dice medio por volumen: {np.mean(volume_dice):.4f}")

## 6. Grad-CAM para el canal foreground

MONAI aplica Grad-CAM usando el índice del canal de salida como `class_idx`. Aquí fijamos el canal 1 (bazo) de los logits por píxel; el score agregado resume la evidencia espacial de foreground. No es una explicación de una región concreta ni sustituye la máscara o la métrica Dice. Para una atribución de ROI habría que definir explícitamente un score limitado a esa ROI.

In [ ]:
case = validation_cases[0]
volume = volume_transform(case)
image_volume = volume["image"].as_tensor()[0].cpu().numpy()
label_volume = (volume["label"].as_tensor()[0].cpu().numpy() > 0).astype(np.uint8)
positive_slices = np.flatnonzero(label_volume.sum(axis=(0, 1)) > 0)
slice_index = int(positive_slices[len(positive_slices) // 2])
slice_image = torch.from_numpy(image_volume[:, :, slice_index][None, None]).float()
slice_label = torch.from_numpy(label_volume[:, :, slice_index][None, None]).float()
slice_image = F.interpolate(slice_image, size=(128, 128), mode="bilinear", align_corners=False).to(device)
slice_label = F.interpolate(slice_label, size=(128, 128), mode="nearest")[0, 0].numpy()
with torch.no_grad():
    slice_prediction = model(slice_image).argmax(dim=1)[0].cpu().numpy()

candidate_layers = [
    (name, module)
    for name, module in model.named_modules()
    if isinstance(module, torch.nn.Conv2d) and module.out_channels != 2
]
target_layer = candidate_layers[-1][0]
print("Capa objetivo Grad-CAM:", target_layer)
gradcam = GradCAM(
    nn_module=model,
    target_layers=target_layer,
    postprocessing=lambda activation: activation,
)
cam = gradcam(slice_image, class_idx=1)[0, 0].detach().cpu().numpy()
cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(slice_image[0, 0].detach().cpu(), cmap="gray"); axes[0].set_title("CT")
axes[1].imshow(slice_label); axes[1].set_title("Máscara real")
axes[2].imshow(slice_prediction); axes[2].set_title("Predicción")
axes[3].imshow(slice_image[0, 0].detach().cpu(), cmap="gray")
axes[3].imshow(cam, cmap="jet", alpha=0.4); axes[3].contour(slice_label, levels=[0.5], colors="white", linewidths=0.8)
axes[3].set_title("Grad-CAM foreground")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## Para discutir

1. ¿La predicción coincide con la máscara en los bordes del bazo? ¿Qué resume y qué oculta Dice?
2. ¿Grad-CAM se superpone con el órgano? ¿Una superposición demuestra que la red segmentó correctamente?
3. ¿Qué diferencia hay entre el mapa de atribución, el logit espacial y la máscara binaria?
4. ¿Qué limitaciones aparecen al entrenar con cortes 2D de estudios 3D?

**Conclusión:** segmentación necesita etiquetas espaciales reales; Grad-CAM aporta una atribución complementaria y más gruesa, no una segunda segmentación ni una explicación causal.